In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# !pip install torchopt
import torchopt


In [ ]:
def test_gamma():
    class Rollout:
        @staticmethod
        def get():
            out = torch.empty(5, 2)
            out[:, 0] = torch.randn(5)
            out[:, 1] = 0.1 * torch.ones(5)
            label = torch.arange(0, 10)
            return out.view(10, 1), F.one_hot(label, 10)

        @staticmethod
        def rollout(trajectory, gamma):
            out = [trajectory[-1]]
            for i in reversed(range(9)):
                out.append(trajectory[i] + gamma[i] * out[-1].clone().detach_())
            out.reverse()
            return torch.hstack(out).view(10, 1)

    class ValueNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(10, 1)

        def forward(self, x):
            return self.fc(x)

    torch.manual_seed(0)
    inner_iters = 1
    outer_iters = 10000
    net = ValueNetwork()
    inner_optimizer = torchopt.MetaSGD(net, lr=5e-1)
    gamma = torch.zeros(9, requires_grad=True)
    meta_optimizer = torchopt.SGD([gamma], lr=5e-1)
    net_state = torchopt.extract_state_dict(net)
    for i in range(outer_iters):
        for _ in range(inner_iters):
            trajectory, state = Rollout.get()
            backup = Rollout.rollout(trajectory, torch.sigmoid(gamma))
            pred_value = net(state.float())

            loss = F.mse_loss(pred_value, backup)
            inner_optimizer.step(loss)

        trajectory, state = Rollout.get()
        pred_value = net(state.float())
        backup = Rollout.rollout(trajectory, torch.ones_like(gamma))

        loss = F.mse_loss(pred_value, backup)
        meta_optimizer.zero_grad()
        loss.backward()
        meta_optimizer.step()
        torchopt.recover_state_dict(net, net_state)
        if i % 100 == 0:
            with torch.no_grad():
                print(f'epoch {i} | gamma: {torch.sigmoid(gamma)}')


if __name__ == '__main__':
    test_gamma()

epoch 0 | gamma: tensor([0.5109, 0.5178, 0.5082, 0.5128, 0.5077, 0.5125, 0.5015, 0.5001, 0.5015])
epoch 100 | gamma: tensor([0.5333, 0.5030, 0.3951, 0.1920, 0.4334, 0.2817, 0.4392, 0.3030, 0.5314])
epoch 200 | gamma: tensor([0.4497, 0.3116, 0.3772, 0.1494, 0.3971, 0.1975, 0.4109, 0.2341, 0.5326])
epoch 300 | gamma: tensor([0.4252, 0.2650, 0.3501, 0.1044, 0.3669, 0.1420, 0.3857, 0.1917, 0.5287])
epoch 400 | gamma: tensor([0.4515, 0.2613, 0.3739, 0.1028, 0.3846, 0.1270, 0.3764, 0.1277, 0.5543])
epoch 500 | gamma: tensor([0.4361, 0.2179, 0.3654, 0.0850, 0.3817, 0.1171, 0.3761, 0.1248, 0.5559])
epoch 600 | gamma: tensor([0.4335, 0.1809, 0.3671, 0.0710, 0.3851, 0.1005, 0.3711, 0.0957, 0.5709])
epoch 700 | gamma: tensor([0.4416, 0.1506, 0.3853, 0.0654, 0.4115, 0.1035, 0.3784, 0.0780, 0.5918])
epoch 800 | gamma: tensor([0.4430, 0.1421, 0.3907, 0.0644, 0.4147, 0.0992, 0.3755, 0.0683, 0.5970])
epoch 900 | gamma: tensor([0.4224, 0.0819, 0.4067, 0.0587, 0.4215, 0.0834, 0.3927, 0.0664, 0.6148])
ep